In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
import os
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from PIL import Image
from rdkit.Chem.Draw import SimilarityMaps
import os
import io


In [2]:
from transformers import AutoTokenizer, AutoModel
import selfies as sf
import torch
import torch.nn.functional as F
from typing import Union
from tqdm import tqdm

_tokenizer = AutoTokenizer.from_pretrained("ibm/materials.selfies-ted")
_model = AutoModel.from_pretrained("ibm/materials.selfies-ted")
_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_model = _model.to(_device)

def smiles_to_selfies(smiles_list: list[str]) -> list[str]:
    """Convert a list of SMILES to tokenizer-ready SELFIES strings."""
    result = []
    for smi in smiles_list:
        sel = sf.encoder(smi)
        sel = sel.replace("][", "] [")  # space-separate tokens
        result.append(sel)
    return result

@torch.no_grad()
def encode(
    smiles: Union[str, list[str]],
    batch_size: int = 64,
    max_length: int = 128,
) -> torch.Tensor:
    """
    Encode SMILES string(s) into mean-pooled embeddings.

    Args:
        smiles:     A single SMILES string or a list of them.
        batch_size: Number of molecules to process per forward pass.
        max_length: Max token length (truncated/padded to this).

    Returns:
        embeddings: Tensor of shape (N, hidden_dim) on CPU.
    """
    if isinstance(smiles, str):
        smiles = [smiles]

    selfies_list = smiles_to_selfies(smiles)
    all_embeddings = []

    _model.eval()
    for i in tqdm(range(0, len(selfies_list), batch_size)):
        batch = selfies_list[i : i + batch_size]

        tokens = _tokenizer(
            batch,
            return_tensors="pt",
            max_length=max_length,
            truncation=True,
            padding="max_length",
        )
        input_ids = tokens["input_ids"].to(_device)
        attention_mask = tokens["attention_mask"].to(_device)

        outputs = _model.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state  # (B, L, D)

        # Mean pooling over non-padding tokens
        mask_expanded = attention_mask.unsqueeze(-1).expand(hidden.size()).float()
        sum_embeddings = torch.sum(hidden * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        embeddings = sum_embeddings / sum_mask  # (B, D)

        all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings, dim=0)  # (N, D)


In [3]:
data = pd.read_csv("sol_data_organic.csv")
data.head()

,Cation,Anion,Capacity,Cation_SMILES,Anion_SMILES
0,benzyl-triphenyl-phosphonium,"(2r,3r)-2,3-dihydroxybutanedioate",2.076798,C(c1ccccc1)[P+](c2ccccc2)(c3ccccc3)c4ccccc4,O[C@H]([C@@H](O)C([O-])=O)C([O-])=O
1,benzyl-triphenyl-phosphonium,1-butanesulfonate,7.674301,C(c1ccccc1)[P+](c2ccccc2)(c3ccccc3)c4ccccc4,CCCC[S]([O-])(=O)=O
2,benzyl-triphenyl-phosphonium,1-hexanesulfonate,7.434469,C(c1ccccc1)[P+](c2ccccc2)(c3ccccc3)c4ccccc4,CCCCCC[S]([O-])(=O)=O
3,benzyl-triphenyl-phosphonium,2-(2-methoxyethoxy)ethylsulfate,1.678763,C(c1ccccc1)[P+](c2ccccc2)(c3ccccc3)c4ccccc4,COCCOCCO[S]([O-])(=O)=O
4,benzyl-triphenyl-phosphonium,"2-hydroxy-1,2,3-propanetricarboxylate",1.232528,C(c1ccccc1)[P+](c2ccccc2)(c3ccccc3)c4ccccc4,OC(CC([O-])=O)(CC([O-])=O)C([O-])=O


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

class SolubilityModel(nn.Module):
    def __init__(self, enc_out_dim, hidden_dim=128):
        super().__init__()
        self.cation_encoder = nn.Linear(enc_out_dim, hidden_dim)
        self.anion_encoder = nn.Linear(enc_out_dim, hidden_dim)
        
        self.fc1 = nn.Linear(hidden_dim * 4, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # Split into cation and anion
        cation_emb = self.cation_encoder(x[:, :x.shape[1]//2])
        anion_emb = self.anion_encoder(x[:, x.shape[1]//2:])
        interaction = cation_emb * anion_emb # Element-wise interaction
        addition = cation_emb + anion_emb # Element-wise addition

        # Combine
        combined = torch.cat([cation_emb.squeeze(1), anion_emb.squeeze(1), interaction.squeeze(1), addition.squeeze(1)], dim=1)
        
        x = self.relu(self.fc1(combined))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
    def predict(self, x):
        self.eval()
        with torch.no_grad():
            return self.forward(x).cpu().numpy()

In [5]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, device):
    model.to(device)
    losses = []

    for epoch in range(num_epochs):
        model.train()
        train_total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_total_loss += loss.item() * X_batch.size(0)

        model.eval()
        val_total_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch).squeeze()
                loss = criterion(outputs, y_batch)
                val_total_loss += loss.item() * X_batch.size(0)

        train_loss = train_total_loss / len(train_loader.dataset)
        val_loss = val_total_loss / len(val_loader.dataset)

        epoch_losses = {
            "train_loss": train_loss,
            "val_loss": val_loss
        }
        losses.append(epoch_losses)
        
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    return losses, model


In [6]:
def build_embedding_cache(data: pd.DataFrame) -> dict[str, np.ndarray]:
    """
    Pre-compute and cache embeddings for every unique SMILES in the dataset.
    Call this ONCE before any split, then pass the cache to create_zero_shot_split.
    """
    all_cation_smiles = data['Cation_SMILES'].unique().tolist()
    all_anion_smiles  = data['Anion_SMILES'].unique().tolist()
    all_smiles = list(set(all_cation_smiles + all_anion_smiles))

    print(f"Encoding {len(all_smiles)} unique SMILES...")
    embs = encode(all_smiles)           # single batched forward pass
    embs_np = embs.numpy()              # (N, D) numpy

    return {smi: embs_np[i] for i, smi in enumerate(all_smiles)}

def create_zero_shot_split(
    data,
    embedding_cache: dict,             # <-- pass the cache in
    test_ratio=0.1,
    val_ratio=0.1,
    seed=None,
):
    np.random.seed(seed)
    unique_cations = data['Cation'].unique()
    unique_anions  = data['Anion'].unique()

    test_cations = np.random.choice(unique_cations, size=int(len(unique_cations)*test_ratio), replace=False)
    test_anions  = np.random.choice(unique_anions,  size=int(len(unique_anions)*test_ratio),  replace=False)

    unique_cations = unique_cations[~np.isin(unique_cations, test_cations)]
    unique_anions  = unique_anions[ ~np.isin(unique_anions,  test_anions)]

    val_cations = np.random.choice(unique_cations, size=int(len(unique_cations)*val_ratio), replace=False)
    val_anions  = np.random.choice(unique_anions,  size=int(len(unique_anions)*val_ratio),  replace=False)

    unique_cations = unique_cations[~np.isin(unique_cations, val_cations)]
    unique_anions  = unique_anions[ ~np.isin(unique_anions,  val_anions)]
    train_cations, train_anions = unique_cations, unique_anions

    train_data = data[data['Cation'].isin(train_cations) & data['Anion'].isin(train_anions)]
    test_data  = data[data['Cation'].isin(test_cations)  & data['Anion'].isin(test_anions)]
    val_data   = data[data['Cation'].isin(val_cations)   & data['Anion'].isin(val_anions)]

    def lookup(df, col):
        """Stack cached embeddings for a column of SMILES strings."""
        return np.vstack([embedding_cache[s] for s in df[col]])

    def make_loader(df, shuffle):
        X = np.hstack([lookup(df, 'Cation_SMILES'), lookup(df, 'Anion_SMILES')])
        y = df['Capacity'].apply(lambda x: np.log10(x) if x > 0 else 0).values
        ds = TensorDataset(
            torch.tensor(X, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
        )
        return DataLoader(ds, batch_size=64, shuffle=shuffle)

    return make_loader(train_data, True), make_loader(test_data, False), make_loader(val_data, False)

In [7]:
num_epochs = 5
device = 'cuda:0'
# Train multiple models with different splits
models = []
cache = build_embedding_cache(data)
for seed in range(5):
    train_loader, test_loader, val_loader = create_zero_shot_split(data, cache, seed=seed)
    
    # Sample one datapoint for input dim
    input_dim = next(iter(train_loader))[0].shape[1]
    print(input_dim)
    model = SolubilityModel(input_dim, hidden_dim=32)
    model.compile()
    criterion = nn.MSELoss()

    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    _, model = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, device)
    models.append(model)
    
# Ensemble predictions
def ensemble_predict(models, X):
    preds = [model.predict(X) for model in models]
    return np.mean(preds, axis=0)

Encoding 221 unique SMILES...


100%|██████████| 4/4 [00:01<00:00,  3.15it/s]


2048


E0303 15:20:51.061000 2339180 torch/_subclasses/fake_tensor.py:2388] [0/0] failed while attempting to run meta for aten.mm.default
E0303 15:20:51.061000 2339180 torch/_subclasses/fake_tensor.py:2388] [0/0] Traceback (most recent call last):
E0303 15:20:51.061000 2339180 torch/_subclasses/fake_tensor.py:2388] [0/0]   File "/media/bose/data4/aadit/.venv/lib/python3.10/site-packages/torch/_subclasses/fake_tensor.py", line 2384, in _dispatch_impl
E0303 15:20:51.061000 2339180 torch/_subclasses/fake_tensor.py:2388] [0/0]     r = func(*args, **kwargs)
E0303 15:20:51.061000 2339180 torch/_subclasses/fake_tensor.py:2388] [0/0]   File "/media/bose/data4/aadit/.venv/lib/python3.10/site-packages/torch/_ops.py", line 723, in __call__
E0303 15:20:51.061000 2339180 torch/_subclasses/fake_tensor.py:2388] [0/0]     return self._op(*args, **kwargs)
E0303 15:20:51.061000 2339180 torch/_subclasses/fake_tensor.py:2388] [0/0]   File "/media/bose/data4/aadit/.venv/lib/python3.10/site-packages/torch/_prims_c

TorchRuntimeError: Failed running call_function <built-in function linear>(*(FakeTensor(..., device='cuda:0', size=(64, 1024)), Parameter(FakeTensor(..., device='cuda:0', size=(32, 2048), requires_grad=True)), Parameter(FakeTensor(..., device='cuda:0', size=(32,), requires_grad=True))), **{}):
a and b must have same reduction dim, but got [64, 1024] X [2048, 32].

from user code:
   File "/tmp/ipykernel_2339180/485973669.py", line 21, in forward
    cation_emb = self.cation_encoder(x[:, :x.shape[1]//2])
  File "/media/bose/data4/aadit/.venv/lib/python3.10/site-packages/torch/nn/modules/linear.py", line 125, in forward
    return F.linear(input, self.weight, self.bias)

Set TORCH_LOGS="+dynamo" and TORCHDYNAMO_VERBOSE=1 for more information


You can suppress this exception and fall back to eager by setting:
    import torch._dynamo
    torch._dynamo.config.suppress_errors = True


In [21]:
# prediction on test set
model.eval()
predictions = []
with torch.no_grad():
    for X_batch, _ in test_loader:
        outputs = model(X_batch).squeeze()
        predictions.extend(outputs.cpu().numpy())

r2 = r2_score(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
print(f"Test R2: {r2:.4f}, Test MSE: {mse:.4f}")

TorchRuntimeError: Failed running call_function <built-in function linear>(*(FakeTensor(..., size=(s0, 1024)), Parameter(FakeTensor(..., device='cuda:0', size=(32, 1024), requires_grad=True)), Parameter(FakeTensor(..., device='cuda:0', size=(32,), requires_grad=True))), **{}):
Unhandled FakeTensor Device Propagation for aten.mm.default, found two different devices cpu, cuda:0

from user code:
   File "/tmp/ipykernel_2336609/1470827509.py", line 23, in forward
    cation_emb = self.cation_encoder(x[:, :x.shape[1]//2])
  File "/media/bose/data4/aadit/.venv/lib/python3.10/site-packages/torch/nn/modules/linear.py", line 125, in forward
    return F.linear(input, self.weight, self.bias)

Set TORCH_LOGS="+dynamo" and TORCHDYNAMO_VERBOSE=1 for more information


You can suppress this exception and fall back to eager by setting:
    import torch._dynamo
    torch._dynamo.config.suppress_errors = True


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, predictions, alpha=0.5, label='Predictions')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('True Log Capacity')
plt.ylabel('Predicted Log Capacity')
plt.title('Neural Network Model Predictions (Test Set)')
plt.legend()
plt.show()

NameError: name 'y_test' is not defined

<Figure size 800x600 with 0 Axes>